# Behavioral Data Visualization

Created on Sun Aug 18 19:53:49 2024

Author: emmaodom

In [ ]:
import numpy as np
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import os 
import glob
import re
import shutil
from scipy.signal import butter, filtfilt
from scipy.signal import medfilt
from scipy.signal import savgol_filter

## Data Loading Functions

In [ ]:
def get_txt_path(directory):
    '''
    This function returns any file under directory that has RoiSet.zip in the filename

    Parameters
    ----------
    directory : str
        The path to the directory to search.

    Returns
    -------
    roi_set_zip : str or None
        The file path to the RoiSet.zip file, or None if not found.
    '''
    txt_files = glob.glob(os.path.join(directory, '**', '*.txt'), recursive=True)
    if len(txt_files) == 0:
        print("No .txt files found.")
        return None
    return txt_files

def get_txt_df(filepath):
    if os.path.getsize(filepath) == 0:
        print(f"Skipping {filepath}: File is empty")
        return None

    df = pd.read_csv(filepath, sep='-', header=None, on_bad_lines='skip')
    df = df.reset_index()
    df.drop(labels=['index',2,3],axis=1,inplace=True)
    df.rename(columns={0: 'Timestamp', 1: 'Print'}, inplace=True)
    
    df['Print'] = df['Print'].astype(str)
    df = df[~df['Print'].str.contains('Solenoid pin state')]
    df['Sensor'] = df['Print'].str.extract(r'(lick|bar)', expand=False)
    df['Value'] = df['Print'].str.extract(r'(\d+)', expand=False).astype(float)
    df['Solenoid'] = df['Print'].str.contains('Solenoid Activated').astype(bool)
    df.drop(labels=['Print'],axis=1,inplace=True)
    
    df = df[~(df['Sensor'].isna() & (df['Solenoid'] == 0))]
    df = convert_timestamps(df)
    df = lick_detection(df)
    return df

## Data Processing Functions

In [ ]:
def convert_timestamps(df):
    df['Timestamp'] = df['Timestamp'].str.strip()
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%H:%M:%S.%f')
    return df

def lick_detection(df, sensor='lick', percentile=0.1, m=2, min_val=15):
    sensor_df = df[df['Sensor'] == sensor]
    baseline_value = sensor_df['Value'].quantile(percentile)
    stdev_value = sensor_df['Value'].std()
    threshold = baseline_value + m * stdev_value
    
    df['Lick_Detected'] = False
    df.loc[df['Sensor'] == sensor, 'Lick_Detected'] = sensor_df['Value'] > threshold
    df.loc[df['Sensor'] == sensor, 'Lick_Detected'] = sensor_df['Value'] > min_val
    return df

## Signal Processing Functions

In [ ]:
def downsample(data, factor):
    return data[::factor]

def butter_lowpass_filter(data, cutoff, fs, order=5):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

def median_filter(data, kernel_size=3):
    return medfilt(data, kernel_size=kernel_size)

def get_sampling_rate(df):
    if not pd.api.types.is_datetime64_any_dtype(df['Timestamp']):
        df['Timestamp'] = df['Timestamp'].str.strip()
        df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%H:%M:%S.%f', errors='coerce')
    
    df['Time_Diff'] = df.groupby('Sensor')['Timestamp'].diff()
    sampling_freq = df.groupby('Sensor')['Time_Diff'].apply(lambda x: 1 / x.mean().total_seconds())
    df.drop(['Time_Diff'], axis=1, inplace=True, errors='ignore')
    print(sampling_freq)
    return sampling_freq

## Visualization Functions

In [ ]:
import matplotlib.dates as mdates

def basic_plot(df, animal_ID, date, stage, dots=True, which_dots='Solenoid'):
    lick_df = df[df['Sensor']=='lick'].iloc[::10, :]
    bar_df = df[df['Sensor']=='bar'].iloc[::40, :]
    
    sns.lineplot(data=lick_df, x='Timestamp', y='Value', label='lick')
    sns.lineplot(data=bar_df, x='Timestamp', y='Value', label='bar')
    
    if dots:
        dots_df = df[df[which_dots]==True]
        scale = 1.2*bar_df['Value'].max()
        y_ = scale*np.ones_like(dots_df['Value'])
        plt.scatter(dots_df['Timestamp'], y_, color='red', s=2, label=which_dots, zorder=5)
    
    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    plt.xticks(rotation=45)
    plt.title(f"{stage} Animal {animal_ID} - {date}")
    plt.legend()
    plt.show()

def smoothed_plot(df, animal_ID, date, stage, order=2, kernel_size=30, 
                 dots=True, which_dots='Solenoid', save=False, directory='none'):
    lick_df = df[df['Sensor'] == 'lick'].iloc[::10, :]
    bar_df = df[df['Sensor'] == 'bar'].iloc[::40, :]
    
    lick_df['Smoothed_Value'] = savgol_filter(lick_df['Value'], window_length=7, polyorder=2)
    bar_df['Smoothed_Value'] = median_filter(bar_df['Value'], kernel_size=kernel_size)
    
    sns.lineplot(data=lick_df, x='Timestamp', y='Smoothed_Value', label='lick')
    sns.lineplot(data=bar_df, x='Timestamp', y='Smoothed_Value', label='bar')
    
    if dots:
        dots_df = df[df[which_dots]==True]
        scale = 1.2*bar_df['Value'].max()
        y_ = scale*np.ones_like(dots_df['Value'])
        plt.scatter(dots_df['Timestamp'], y_, color='red', s=2, label=which_dots, zorder=5)
    
    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    plt.xticks(rotation=45)
    plt.title(f"{stage} Animal {animal_ID} - {date}\nbar: median_filter lick: savgol_filter")
    plt.legend()
    
    if save:
        save_path = directory + '/pdf/' + f'{animal_ID}_smoothed_lick_bar_plot.pdf'
        plt.savefig(save_path, format='pdf')
    plt.show()

## Trial Analysis Functions

In [ ]:
def trial_lick_detect(row, df):
    mask = (df['Timestamp'] >= row['trial_start']) & (df['Timestamp'] <= row['trial_end'])
    return int(df.loc[mask,'Lick_Detected'].any())

def trial_performance(filepath, min_bar_hold=1.5, max_trial_dur=10):
    df = get_txt_df(filepath)
    trial_start = df[df['Solenoid']==True]['Timestamp'].reset_index(drop=True)
    trial_dur = trial_start.diff().shift(-1)
    trial_dur = trial_dur.dt.total_seconds().fillna(min_bar_hold)
    
    trial_end_est = trial_start + pd.to_timedelta(trial_dur, unit='s')
    trial_end_max = trial_start + pd.to_timedelta(max_trial_dur, unit='s')
    trial_end = trial_end_est.combine(trial_end_max, min)
    
    trial_df = pd.DataFrame({
        'trial_start': trial_start,
        'trial_end': trial_end
    })
    
    trial_df['Lick_Detected'] = trial_df.apply(trial_lick_detect, axis=1, df=df)
    return trial_df

def sess_performance(filepath, min_bar_hold=1.5, max_trial_dur=10):
    df = get_txt_df(filepath)
    trial_df = trial_performance(filepath, min_bar_hold, max_trial_dur)
    total_trials = len(trial_df)
    lick_trials = trial_df['Lick_Detected'].sum()
    
    dur = (df['Timestamp'].max() - df['Timestamp'].min())
    session_duration = dur.total_seconds()/60
    
    filename = os.path.basename(filepath)
    match = re.match(r"(\d{3,4}[A-Z])_(.+?)_(\d{8})_(\d{6})", filename)
    if match:
        animal_ID, stage, date, start_time = match.groups()
    else:
        raise ValueError(f"Filename '{filename}' does not match pattern")
    
    start_time = pd.to_datetime(start_time, format='%H%M%S')
    if start_time.hour < 6:
        date = pd.to_datetime(date, format='%Y%m%d') - pd.Timedelta(days=1)
    else:
        date = pd.to_datetime(date, format='%Y%m%d')
    
    return pd.DataFrame({
        'filename': [filename],
        'animal_ID': [animal_ID],
        'stage': [stage],
        'date': [date.strftime('%Y-%m-%d')],
        'start': [start_time.strftime('%H:%M:%S')],
        'total_trials': [total_trials],
        'lick_trials': [lick_trials],
        'duration_min': [session_duration]
    })

## Example Usage

In [ ]:
# Example file path
filepath = '/path/to/your/data/file.txt'

# Load and process data
df = get_txt_df(filepath)

# Get sampling rate
sr = get_sampling_rate(df)

# Create visualizations
basic_plot(df, animal_ID='test', date='20240101', stage='Lick', 
          dots=True, which_dots='Lick_Detected')

smoothed_plot(df, animal_ID='test', date='20240101', stage='Lick',
             kernel_size=99, dots=True, which_dots='Lick_Detected')